In [1]:
cd ..

/Users/martinsevilla/Downloads/congestion-bogota


In [9]:
from src.sparse_dataloader import TrafficDataset
from torch_geometric.loader import DataLoader

%load_ext autoreload
%autoreload 2

%load_ext line_profiler

# Reload to get the optimized version
import importlib
import src.sparse_dataloader
importlib.reload(src.sparse_dataloader)
from src.sparse_dataloader import TrafficDataset

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


In [10]:
processed_data_dir = "data/datos_procesados"
sparse_data_dir = f"{processed_data_dir}/sparse_datasets"
graph_filename = f"{processed_data_dir}/conectivity_graph/road_graph.pickle"
static_data_filename = f"{sparse_data_dir}/static_dataset.h5"
dynamic_data_filenames = [
    f"{sparse_data_dir}/coi_points_dataset.h5",
    f"{sparse_data_dir}/coi_tracts_dataset.h5",
    f"{sparse_data_dir}/irregularities_dataset.h5",
    f"{sparse_data_dir}/events_dataset.h5",
    f"{sparse_data_dir}/incidents_dataset.h5",
]
metadata_filenames = [
    f"{sparse_data_dir}/coi_points_dataset_metadata.pkl",
    f"{sparse_data_dir}/coi_tracts_dataset_metadata.pkl",
    f"{sparse_data_dir}/irregularities_dataset_metadata.pkl",
    f"{sparse_data_dir}/events_dataset_metadata.pkl",
    f"{sparse_data_dir}/incidents_dataset_metadata.pkl",
]
target_data_filename = f"{sparse_data_dir}/jams_dataset.h5"
in_timesteps = 150
out_timesteps = 50

traffic_dataset = TrafficDataset(
    graph_path=graph_filename,
    static_data_path=static_data_filename,
    dynamic_data_paths=dynamic_data_filenames,
    metadata_paths=metadata_filenames,
    target_data_path=target_data_filename,
    in_timesteps=in_timesteps,
    out_timesteps=out_timesteps,
    future_features = [
        "coi_puntos",
        "coi_tramo",
        "AGLOMERACION",
        "EMERGENCIA",
        "MANIFESTACION",
    ],
    node_batch_size=500,
)

In [11]:
batch_size = 16

traffic_dataloader = DataLoader(
    dataset=traffic_dataset,
    batch_size=batch_size, # This now batches the Data objects
    shuffle=True,
    num_workers=0 
)

In [5]:
print(f"\nDataset size: {len(traffic_dataset)} samples")
print(f"Number of batches: {len(traffic_dataloader)}")
print("-" * 40)
print("-" * 40)


Dataset size: 9747423 samples
Number of batches: 609214
----------------------------------------
----------------------------------------


In [6]:
# print(f"\nDataset size: {len(traffic_dataset)} samples")
# print(f"Number of batches: {len(traffic_dataloader)}")
# print("-" * 40)
# print("-" * 40)
# # Get a single batch
# sample_batch = next(iter(traffic_dataloader))

# # Inspect the batch
# print(f"Edge index shape (batch_size, 2, num_edges):")
# print(f"{sample_batch['edge_index'].shape}")
# print()
# print(f"Static features shape (batch_size, num_nodes, num_static_features):")
# print(f"{sample_batch['x_static'].shape}")
# print()
# print(f"Past dynamic features shape (batch_size, num_nodes, in_timesteps, num_dynamic_features):")
# print(f"{sample_batch['x_dynamic_past'].shape}")
# print()
# print(f"Future dynamic features shape (batch_size, num_nodes, out_timesteps, num_dynamic_features):")
# print(f"{sample_batch['x_dynamic_future'].shape}")
# print()
# print(f"Target shape (batch_size, num_nodes, out_timesteps):")
# print(f"{sample_batch['y'].shape}")

In [12]:
%lprun -f TrafficDataset.__getitem__ -u 1e-06 [next(iter(traffic_dataloader)) for _ in range(1)]

Timer unit: 1e-06 s

Total time: 1.14238 s
File: /Users/martinsevilla/Downloads/congestion-bogota/src/sparse_dataloader.py
Function: TrafficDataset.__getitem__ at line 352

Line #      Hits         Time  Per Hit   % Time  Line Contents
   352                                               def __getitem__(self, idx) -> Data:
   353                                                   """
   354                                                   Returns a single sample, which may be a subset of nodes for a specific
   355                                                   time window if node_batch_size is set.
   356                                                   """
   357        16       2997.0    187.3      0.3          time_idx, node_indices = self._get_time_and_node_indices(idx)
   358                                                   # Past relevant time indices
   359        16          7.0      0.4      0.0          input_start_time = time_idx
   360        16         11.0      0.7 

In [13]:
import time
num_epochs = 3  # Reduced for testing

for epoch in range(num_epochs):
    epoch_start = time.perf_counter()
    total_loss = 0.0
    prev_end = epoch_start

    for step, batch in enumerate(traffic_dataloader):
        iter_start = time.perf_counter()
        data_wait_s = iter_start - prev_end
        print(f"Batch {step}: dataloader wait = {data_wait_s*1000:.1f} ms")
        prev_end = time.perf_counter()
        if step >= 3:  # Only test first few batches
            break

Batch 0: dataloader wait = 1310.3 ms
Batch 1: dataloader wait = 1113.7 ms
Batch 1: dataloader wait = 1113.7 ms
Batch 2: dataloader wait = 1032.9 ms
Batch 2: dataloader wait = 1032.9 ms
Batch 3: dataloader wait = 1072.0 ms
Batch 3: dataloader wait = 1072.0 ms
Batch 0: dataloader wait = 1294.0 ms
Batch 0: dataloader wait = 1294.0 ms
Batch 1: dataloader wait = 1009.5 ms
Batch 1: dataloader wait = 1009.5 ms
Batch 2: dataloader wait = 1000.9 ms
Batch 2: dataloader wait = 1000.9 ms
Batch 3: dataloader wait = 1094.0 ms
Batch 3: dataloader wait = 1094.0 ms
Batch 0: dataloader wait = 1231.5 ms
Batch 0: dataloader wait = 1231.5 ms
Batch 1: dataloader wait = 1195.4 ms
Batch 1: dataloader wait = 1195.4 ms
Batch 2: dataloader wait = 1002.1 ms
Batch 2: dataloader wait = 1002.1 ms
Batch 3: dataloader wait = 1114.6 ms
Batch 3: dataloader wait = 1114.6 ms


## Compare sorted search vs. naive approach


In [22]:
import h5py
import numpy as np

file_path = traffic_dataset.dynamic_data_paths[2]
idx = 100


file_indices = np.zeros((traffic_dataset.total_timesteps_in_data - 1, 2), dtype=int)
with h5py.File(file_path, "r") as f:
    # Reading the entire time_idx column can be slow
    time_column = f["data"][:, 1]
    file_indices[:, 0] = np.searchsorted(time_column, np.arange(traffic_dataset.total_timesteps_in_data - 1), side='left')
    file_indices[:, 1] = np.searchsorted(time_column, np.arange(traffic_dataset.total_timesteps_in_data - 1), side='right')

def demo_no_sorting(idx):
    input_start_time = idx
    input_end_time = input_start_time + 50
    with h5py.File(file_path, "r") as f:
        # Reading the entire time_idx column can be slow
        all_time_idx = f["data"][:, 1]
        # Find the rows that fall within our time window
        # TODO: If the data is sorted by time, np.searchsorted can make this much faster
        rows_in_window_mask = (all_time_idx >= input_start_time) & (
            all_time_idx < input_end_time
        )
        relevant_data = f["data"][rows_in_window_mask]
        del all_time_idx, rows_in_window_mask  # Free memory
    return relevant_data


def demo_sorted(idx):
    input_start_time = idx
    input_end_time = input_start_time + 50
    with h5py.File(file_path, "r") as f:
        start_index = file_indices[input_start_time, 0]
        end_index = file_indices[input_end_time, 1]
        relevant_data = f["data"][start_index:end_index]
    return relevant_data

relevant_data_no_sorting = demo_no_sorting(idx)
relevant_data_sorted = demo_sorted(idx)

assert (relevant_data_sorted == relevant_data_no_sorting).mean() == 1.0, "Data mismatch between sorted and unsorted methods"

In [12]:
%lprun -f demo_no_sorting demo_no_sorting(idx)

Timer unit: 1e-09 s

Total time: 2.35875 s
File: /var/folders/h9/yn87zkms09g2yrr0r9s65s6h0000gn/T/ipykernel_64228/3345745773.py
Function: demo_no_sorting at line 15

Line #      Hits         Time  Per Hit   % Time  Line Contents
    15                                           def demo_no_sorting(idx):
    16         1       1000.0   1000.0      0.0      input_start_time = idx
    17         1          0.0      0.0      0.0      input_end_time = input_start_time + 50
    18         2     400000.0 200000.0      0.0      with h5py.File(file_path, "r") as f:
    19                                                   # Reading the entire time_idx column can be slow
    20         1 1995617000.0    2e+09     84.6          all_time_idx = f["data"][:, 1]
    21                                                   # Find the rows that fall within our time window
    22                                                   # TODO: If the data is sorted by time, np.searchsorted can make this much faster


In [13]:
%lprun -f demo_sorted demo_sorted(idx)

Timer unit: 1e-09 s

Total time: 0.000815 s
File: /var/folders/h9/yn87zkms09g2yrr0r9s65s6h0000gn/T/ipykernel_64228/3345745773.py
Function: demo_sorted at line 31

Line #      Hits         Time  Per Hit   % Time  Line Contents
    31                                           def demo_sorted(idx):
    32         1       1000.0   1000.0      0.1      input_start_time = idx
    33         1          0.0      0.0      0.0      input_end_time = input_start_time + 50
    34         2     588000.0 294000.0     72.1      with h5py.File(file_path, "r") as f:
    35         1       4000.0   4000.0      0.5          start_index = file_indices[input_start_time, 0]
    36         1       1000.0   1000.0      0.1          end_index = file_indices[input_end_time, 1]
    37         1     220000.0 220000.0     27.0          relevant_data = f["data"][start_index:end_index]
    38         1       1000.0   1000.0      0.1      return relevant_data

In [21]:
print("Improvement of", round(100 - (0.000815 / 2.35875) * 100, 2), "%")

Improvement of 99.97 %


## How does sparsity work?


In [ ]:
import pandas as pd
import numpy as np

rows, cols = 100, 50
data = np.zeros((rows, cols), dtype=int)

# Randomly assign some nonzero values
np.random.seed(42)
num_nonzero = 50
row_idx = np.random.randint(0, rows, num_nonzero)
col_idx = np.random.randint(0, cols, num_nonzero)
values = np.random.randint(1, 100, num_nonzero)
data[row_idx, col_idx] = values

df = pd.DataFrame(data, columns=range(1, cols+1))

df.head(14).style.map(lambda x: 'color: red' if x != 0 else '', subset=df.columns)

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,35,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,41,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Convert to long format for better visualization
df_long = df.reset_index().melt(id_vars='index', var_name='Feature', value_name='Value')
df_long = df_long[df_long['Value'] != 0]
df_long.sort_values(by=['index', 'Feature'], inplace=True)
df_long.head().style.map(lambda x: 'color: red' if x != 0 else '', subset=['Value'])

,index,Feature,Value
1301,1,14,35
3901,1,40,41
702,2,8,11
1702,2,18,5
2306,6,24,14


In [51]:
# Size of df in KB
df_size_kb = df.memory_usage(deep=True).sum() / 1024
print(f"DataFrame size: {df_size_kb:.2f} KB")

# Size of df_long in KB
df_long_size_kb = df_long.memory_usage(deep=True).sum() / 1024
print(f"Long format DataFrame size: {df_long_size_kb:.2f} KB")

DataFrame size: 39.19 KB
Long format DataFrame size: 2.93 KB
